# Model Training & Experimentation Notebook

This notebook is for **experimenting with different models, hyperparameters, and evaluating performance**.

**Key Points:**
- Uses the same training functions from `app/risk_scoring.py` (production code)
- Visualizes model performance, feature importance, and metrics
- Documents hyperparameter tuning and model selection
- Can export best models for production use

**Production Note:** The Streamlit app automatically uses models from `models/` directory. This notebook is for experimentation and evaluation only.


## Setup & Imports


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve
)

# Import our production training functions
from app.risk_scoring import score_transactions, retrain_models
from app.data_loader import load_transaction_data

# Set style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')
sns.set_palette(['#d62728', '#2ca02c'])

print("✅ Setup complete")


## Load Data


In [ ]:
# Load BI data (or full dataset from MySQL)
# Try different paths in case notebook is run from different directories
csv_paths = [
    "data/bi/feat_transactions_risk_bi.csv",  # Default path (from project root)
    "../data/bi/feat_transactions_risk_bi.csv",  # If run from notebooks/
    str(project_root / "data/bi/feat_transactions_risk_bi.csv"),  # Absolute from project root
]

df = None
for csv_path in csv_paths:
    try:
        test_df = load_transaction_data(csv_path)
        if not test_df.empty:
            df = test_df
            print(f"✅ Loaded data from: {csv_path}")
            break
    except Exception as e:
        continue

if df is None or df.empty:
    print("❌ ERROR: No data found!")
    print("\nPlease run the BI data export first:")
    print("  uv run python src/export_bi_data.py")
    print("\nOr ensure the CSV file exists at one of these paths:")
    for path in csv_paths:
        print(f"  - {path}")
    raise FileNotFoundError("Transaction data CSV not found. Please run src/export_bi_data.py first.")

print(f"\n✅ Loaded {len(df):,} transactions")
if 'is_fraud' in df.columns:
    fraud_rate = df['is_fraud'].mean() * 100
    print(f"✅ Fraud rate: {fraud_rate:.2f}%")
    print(f"   Fraud transactions: {df['is_fraud'].sum():,}")
    print(f"   Legitimate transactions: {(df['is_fraud'] == False).sum():,}")
else:
    print("⚠️ Warning: 'is_fraud' column not found")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")


## Train-Test Split

Split data for proper evaluation. We'll train on 80% and test on 20%.


In [ ]:
# Split data for evaluation
# Use 80% for training, 20% for testing

# Check if we have data and fraud labels
if df.empty:
    raise ValueError("Cannot split empty dataset. Please load data first.")
    
if 'is_fraud' not in df.columns:
    raise ValueError("'is_fraud' column not found. Cannot perform stratified split.")

# Check if we have both fraud and non-fraud cases for stratification
fraud_count = df['is_fraud'].sum()
non_fraud_count = (df['is_fraud'] == False).sum()

if fraud_count == 0 or non_fraud_count == 0:
    print("⚠️ Warning: Dataset has only one class. Using non-stratified split.")
    train_df, test_df = train_test_split(
        df, 
        test_size=0.2, 
        random_state=42, 
        stratify=None
    )
else:
    # Stratified split to maintain fraud rate in both splits
    train_df, test_df = train_test_split(
        df, 
        test_size=0.2, 
        random_state=42, 
        stratify=df['is_fraud']
    )

print(f"✅ Training set: {len(train_df):,} transactions ({train_df['is_fraud'].mean()*100:.2f}% fraud)")
print(f"✅ Test set: {len(test_df):,} transactions ({test_df['is_fraud'].mean()*100:.2f}% fraud)")


## Train Models

Train models using the production `score_transactions()` function. This will save models to `models/` directory.


In [ ]:
# Train models on training set
# This uses the production score_transactions() function which will:
# 1. Train ensemble models (Logistic Regression + Random Forest)
# 2. Save models to models/ directory
# 3. Return scored transactions

print("Training models on training set...")
print("This may take 10-30 seconds for large datasets...")

train_scored = score_transactions(train_df)

print("\n✅ Models trained and saved to models/ directory")
print(f"\nTraining set risk score distribution:")
print(train_scored['risk_score'].describe())


## Evaluate on Test Set

Score the test set using the saved models and calculate performance metrics.


In [ ]:
# Score test set (uses saved models from models/ directory)
test_scored = score_transactions(test_df)

# Calculate metrics
y_true = test_scored['is_fraud'].astype(int)
y_pred = (test_scored['risk_score'] >= 0.5).astype(int)  # Threshold at 0.5
y_proba = test_scored['risk_score']

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Legitimate', 'Fraud']))

# ROC AUC
roc_auc = roc_auc_score(y_true, y_proba)
print(f"\nROC AUC Score: {roc_auc:.4f}")


## Visualize Model Performance


In [ ]:
# Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_proba)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## Risk Score Distribution by Fraud Status


In [ ]:
# Distribution of risk scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
fraud_scores = test_scored[test_scored['is_fraud'] == True]['risk_score']
legit_scores = test_scored[test_scored['is_fraud'] == False]['risk_score']

axes[0].hist(legit_scores, bins=50, alpha=0.7, label='Legitimate', color='green')
axes[0].hist(fraud_scores, bins=50, alpha=0.7, label='Fraud', color='red')
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold (0.5)')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Risk Score Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot
score_df = pd.DataFrame({
    'Risk Score': pd.concat([fraud_scores, legit_scores]),
    'Type': ['Fraud'] * len(fraud_scores) + ['Legitimate'] * len(legit_scores)
})
sns.boxplot(data=score_df, x='Type', y='Risk Score', ax=axes[1])
axes[1].set_title('Risk Score by Transaction Type')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Feature Importance (from Random Forest)


In [ ]:
# Load the trained Random Forest model to get feature importance
import pickle
from pathlib import Path

models_dir = Path('../models')
rf_model_path = models_dir / 'rf_model.pkl'

if rf_model_path.exists():
    with open(rf_model_path, 'rb') as f:
        rf_model = pickle.load(f)
    
    # Get feature importance
    feature_names = ['txns_last_24h', 'declined_txns_last_24h', 
                     'merchant_fraud_rate_30d', 'is_high_risk_merchant', 
                     'is_emulator_device']
    
    importances = rf_model.feature_importances_
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    indices = np.argsort(importances)[::-1]
    
    ax.barh(range(len(feature_names)), importances[indices])
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels([feature_names[i] for i in indices])
    ax.set_xlabel('Feature Importance')
    ax.set_title('Random Forest Feature Importance')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print feature importance values
    print("\nFeature Importance:")
    for i, idx in enumerate(indices):
        print(f"  {feature_names[idx]}: {importances[idx]:.4f}")
else:
    print("Model file not found. Train models first.")


## Model Performance by Risk Band


In [ ]:
# Analyze performance by risk band
band_analysis = test_scored.groupby('risk_band').agg({
    'is_fraud': ['count', 'sum', 'mean'],
    'risk_score': 'mean'
}).round(4)

band_analysis.columns = ['Total', 'Fraud Count', 'Fraud Rate', 'Avg Risk Score']
print("Performance by Risk Band:")
print(band_analysis)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by band
band_analysis['Fraud Rate'].plot(kind='bar', ax=axes[0], color=['green', 'orange', 'red'])
axes[0].set_title('Fraud Rate by Risk Band')
axes[0].set_ylabel('Fraud Rate')
axes[0].set_xlabel('Risk Band')
axes[0].grid(True, alpha=0.3)

# Transaction count by band
band_analysis['Total'].plot(kind='bar', ax=axes[1], color=['green', 'orange', 'red'])
axes[1].set_title('Transaction Count by Risk Band')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Risk Band')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Precision-Recall Curve

Useful for imbalanced datasets (fraud is rare).


In [ ]:
# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_true, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, label='Precision-Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find optimal threshold (maximize F1 score)
f1_scores = 2 * (precision * recall) / (precision + recall)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Optimal F1 Score: {optimal_f1:.3f}")
print(f"Precision at optimal: {precision[optimal_idx]:.3f}")
print(f"Recall at optimal: {recall[optimal_idx]:.3f}")


## Export Best Model to Production

If you're satisfied with the model performance, the models are already saved to `models/` directory and will be used by the Streamlit app automatically.

To force retrain with new data or different parameters:
```python
retrain_models(df, force=True)
```
